# All-96-head A/B/C/D exact-effect screen — Colab (zip-unpack)

Unpacks the self-contained package from your Drive and runs the screen. **Boundary:** EXPLORATORY head-effect matrix on DISCOVERY genes. C = *competitor-site perturbation response* (specificity was null — NOT competition-specific). No sealed (chr8/9) run. No biological-function naming. **No full run / no credit spend until you review the benchmark and set `LAUNCH_FULL=True`.**

## 1. Mount Drive + unpack the zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/gfm_head_screen_colab.zip'  # <-- put the zip here
!rm -rf /content/pkg && mkdir -p /content/pkg && unzip -q $ZIP -d /content/pkg
%cd /content/pkg/gfm_head_screen/repo
!pwd && ls

## 2. Install pinned env (torch is Colab's preinstalled CUDA build — not reinstalled)

In [ ]:
!pip install -q -r studies/splicebert_head_context_atlas/colab_package/requirements-colab.txt
import torch, transformers, sys
print('py', sys.version.split()[0], '| torch', torch.__version__, '| tf', transformers.__version__, '| cuda', torch.version.cuda, '|', torch.cuda.get_device_name(0))

## 3. Fetch the two large data files (GTF + hg38) into data/genomes/
The 75 MB SpliceBERT checkpoint is already bundled. hg38 is ~3 GB — to avoid re-downloading every session, copy `data/genomes/` to Drive after the first fetch and symlink it back.

In [ ]:
DRIVE_DATA = '/content/drive/MyDrive/gfm_head_screen_data'  # persistent data cache
import os
if os.path.exists(f'{DRIVE_DATA}/hg38.fa'):
    !mkdir -p data/genomes && ln -sf $DRIVE_DATA/hg38.fa data/genomes/hg38.fa && ln -sf $DRIVE_DATA/hg38.fa.fai data/genomes/hg38.fa.fai 2>/dev/null; ln -sf $DRIVE_DATA/gencode.v50.annotation.gtf.gz data/genomes/gencode.v50.annotation.gtf.gz
    print('using cached data from Drive')
else:
    !bash studies/splicebert_head_context_atlas/colab_package/fetch_data.sh
    !mkdir -p $DRIVE_DATA && cp data/genomes/hg38.fa data/genomes/gencode.v50.annotation.gtf.gz $DRIVE_DATA/ && echo 'cached to Drive for next time'

## 4. Verify hashes (STOP on mismatch)

In [ ]:
import hashlib, json
cfg = json.load(open('studies/splicebert_head_context_atlas/colab_package/config.colab.json'))
def sha(p, n=1<<20):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(n), b''): h.update(b)
    return h.hexdigest()
assert sha(cfg['gtf'])==cfg['gtf_sha256'], 'GTF sha mismatch - STOP'
print('GTF sha OK:', cfg['gtf_sha256'][:16], '| checkpoint present:', os.path.exists(cfg['model']+'/pytorch_model.bin'))

## 5. Mechanical tests + Colab smoke (1 example, all 96 heads, all gates)

In [ ]:
!PYTHONHASHSEED=0 python -m pytest tests/test_competition.py tests/test_head_context_screen.py -q
!PYTHONHASHSEED=0 TOKENIZERS_PARALLELISM=false python scripts/splicebert_head_context_atlas/run_all_head_screen.py \
    --config studies/splicebert_head_context_atlas/colab_package/config.colab.json --n 1 --batch 1 --self-test --out /content/shards_smoke
!python scripts/splicebert_head_context_atlas/validate_head_screen.py --out /content/shards_smoke

## 6. Benchmark (12 examples) — projected full-run GPU-hours + storage

In [ ]:
!PYTHONHASHSEED=0 TOKENIZERS_PARALLELISM=false python scripts/splicebert_head_context_atlas/benchmark_colab.py \
    --config studies/splicebert_head_context_atlas/colab_package/config.colab.json --n 12 --batch 8

## 7. STOP — review the projection
Do not run the next cell until you approve the projected GPU-hours/storage. Then set `LAUNCH_FULL = True`. Output shards go to Drive so a session timeout is resumable.

In [ ]:
LAUNCH_FULL = False  # <-- set True ONLY after you review the benchmark
assert LAUNCH_FULL, 'Full run gated on your GO after benchmark review.'
OUT = '/content/drive/MyDrive/gfm_head_screen_shards'  # resumable across session timeouts
!PYTHONHASHSEED=0 TOKENIZERS_PARALLELISM=false python scripts/splicebert_head_context_atlas/run_all_head_screen.py \
    --config studies/splicebert_head_context_atlas/colab_package/config.colab.json --batch 8 --out $OUT

## 8. Validate + merge -> head-effect matrix + C33 group summary

In [ ]:
!python scripts/splicebert_head_context_atlas/validate_head_screen.py --out $OUT
!python scripts/splicebert_head_context_atlas/merge_head_screen.py --out $OUT
print('matrix + summary in', OUT)